# Oxford Pets: Welche Bildregionen verändern die Unsicherheit?

Dieses eigenständige Experiment nutzt die neue probly-Anbindung in `shapiq_games.probly`.
Ein eingefrorener ImageNet-VGG16 liefert 4.096 Merkmale; drei trainierte Köpfe bilden ein
probly-Subensemble. MC Dropout mit **64 Samples** betrifft ausschließlich einen trainierten Kopf.
Beide Verfahren erklären dieselben gewählten Bildregionen durch **P, TU, AU und EU**,
Shapley-Werte und k-SII bis Ordnung zwei, wenn mindestens zwei Regionen vorhanden sind. Ein zweites Ensemble mit **20 Trainingsbildern pro Rasse** prüft
beispielhaft die Reaktion auf weniger Trainingsdaten.

Die Demo zeigt die Fälle **„Sicher richtig“**, **„Unsicher richtig“** und **„Sicher falsch“**
für das Ensemble mit voller Trainingsmenge. Die Alternative `COHORT_MODE = "seeded"`
zieht Bilder anhand von Rassenlabels und Seed. Mit `n_images=37` zeigt sie ein Bild je Rasse.
Die Diagnosefälle illustrieren einzelne Vorhersagen und regionale Erklärungen.

## 1. Umgebung und Laufzeit

Vorbereitung aus dem Repository-Root: die in der [README](README.md) beschriebenen Extras und
das Projekt-Setup installieren. Im Notebook finden keine Paketinstallationen statt.

Ein frischer Lauf lädt ungefähr **0,8 GB Bildarchive und 0,5 GB VGG-Gewichte**, extrahiert
VGG-Merkmale für 7.349 Originalbilder und bis zu 512 Koalitionen je Diagnosebild und trainiert
sechs kleine Köpfe. Dafür sind mehrere GB freier Speicher und, je nach CPU/GPU, mehrere Minuten bis
Stunden nötig. Bereits passende lokale Merkmale und Köpfe werden wiederverwendet.

`feature_device="auto"` wählt eine verfügbare GPU für die eingefrorene Merkmalsextraktion oder
fällt auf die CPU zurück. Eine explizite Wahl von `"cpu"`, `"mps"` oder `"cuda"` bleibt möglich.
Die konservative Extraktionsbatchgröße ist vier; es findet kein Hardwarebenchmark statt.
Kopftraining und Unsicherheitsberechnung bleiben auf der CPU. Das Notebook setzt keine feste
Threadzahl. Gerätewahl, verfügbare Backends, tatsächliche Threadzahl und Datentypen werden
angezeigt und gespeichert. Hardwareübergreifend identische Bits werden nicht zugesagt.

Daten, Gewichte und Ergebnisse bleiben im ignorierten `data/`, `models/` und `outputs/` dieses
Themenordners. Optionale Pfadvariablen erlauben ausdrücklich angegebene lokale Quellen;
das Notebook benötigt keinen anderen Checkout und keine früheren Notebooks.

In [ ]:
from __future__ import annotations

import json
import sys
import time
from dataclasses import asdict
from importlib.metadata import version
from pathlib import Path
from typing import TYPE_CHECKING, Literal, TypedDict

import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import (
    Markdown,
    display as show_display,
)

from shapiq import ExactComputer, InteractionValues
from shapiq_games.probly import ImageProblyGame, ProblyGame, ProblyPredictor, torch_cpu_context

if TYPE_CHECKING:
    from shapiq_games.typing import Fill

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "shapiq").is_dir())
TOPIC = ROOT / "experiments" / "oxford_pets_uncertainty"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


class SummaryRow(TypedDict):
    image_id: str
    condition: str
    method: str
    true_class: str
    target: str
    target_index: int
    P: float
    TU: float
    AU: float
    EU: float

In [ ]:
%matplotlib inline

In [ ]:
from experiments.oxford_pets_uncertainty.helpers import (
    SUPERPIXEL_TARGET,
    Config,
    RegionMode,
    cached_coalition_features,
    crop_image,
    encode_crops,
    fitted_conditions,
    image_regions,
    load_encoder,
    package_versions,
    prepare_data,
    split_features,
)
from experiments.oxford_pets_uncertainty.reporting import (
    ExplanationResult,
    explanation_plot_limits,
    export_explanations,
    export_summary,
    plot_epistemic_comparison,
    plot_region_explanation,
    plot_test_overview,
)
from experiments.oxford_pets_uncertainty.selection import CohortMode, select_cases

config = Config(n_images=3, feature_device="auto")  # n_images applies to "seeded".
COHORT_MODE: CohortMode = "demo"
REGION_MODE: RegionMode = "trimap"  # "grid" or "superpixels" for other player definitions.
MASK_FILL: Fill = "blur"
DOWNLOAD_DATA = True
DATA_ROOT: Path | None = None
WEIGHTS_ROOT: Path | None = None
EXTERNAL_FEATURES: dict[Literal["trainval", "test"], Path] | None = None
# Optional: {"trainval": Path(...), "test": Path(...)} with documented feature manifests.

run_started = time.perf_counter()
timings: dict[str, float] = {}
output_dir = TOPIC / "outputs"
if REGION_MODE != "grid":
    output_dir /= REGION_MODE
versions = package_versions()
if REGION_MODE == "superpixels":
    versions["scikit-image"] = version("scikit-image")
print("Configuration:", asdict(config), "| cohort:", COHORT_MODE, "| regions:", REGION_MODE)
print("Versions and official source:", versions)

## 2. Feste Datenaufteilung

Der offizielle trainval-Split wird nach Rasse stratifiziert zu 80 %/20 % geteilt (Seed 0).
Die 20er-Teilmengen stammen aus dem Trainingssplit. Der offizielle Testsplit liefert die
Bilder für die spätere Diagnose.
Die VGG-Vorverarbeitung wird aus `VGG16_Weights.IMAGENET1K_V1` übernommen: Resize, Center Crop
auf 224×224 und ImageNet-Normalisierung; keine zufällige Augmentation.

In [ ]:
data = prepare_data(TOPIC, config, data_root=DATA_ROOT, download=DOWNLOAD_DATA)
print(
    "Train / validation / test:",
    len(data["train"]),
    len(data["validation"]),
    len(data["datasets"]["test"]),
)

## 3. Einen Merkmalsraum für alle Köpfe und Verfahren berechnen

VGG bleibt eingefroren und im `eval()`-Modus, einschließlich seiner ursprünglichen Dropouts.
Merkmale werden getrennt von Kopfgewichten gespeichert. Deshalb können Ensemble, MC Dropout,
alle Unsicherheitskomponenten und die Datenknappheitsbedingung denselben teuren Cache verwenden.
Die Normalisierung erfolgt nach dem Zuschneiden; maskierte Bilder werden nicht erneut zugeschnitten.

In [ ]:
started = time.perf_counter()
encoder, runtime = load_encoder(TOPIC, config, weights_root=WEIGHTS_ROOT)
show_display(
    Markdown(
        "**Actual runtime settings**\n\n```json\n" + json.dumps(asdict(runtime), indent=2) + "\n```"
    )
)
features = split_features(TOPIC, data, encoder, config, external_features=EXTERNAL_FEATURES)
timings["feature_preparation_seconds"] = time.perf_counter() - started
print("Frozen feature shapes:", features["trainval"].shape, features["test"].shape)
print(f"Feature preparation: {timings['feature_preparation_seconds'] / 60:.1f} min")

## 4. Köpfe trainieren, echte probly-Subensembles aufbauen

Jeder Kopf: `Dropout(0.5) → Linear(4096, 37)`. AdamW (`lr=0.001`, `weight_decay=0.0001`),
Batchgröße 64, höchstens 30 Epochen, fünf Epochen Patience. Der kleinste Validierungsverlust
wählt den Zustand. probly initialisiert die drei Köpfe unabhängig mit Initialisierungsseed 0.
Trainingsseeds 0/1/2 variieren Dropout und Minibatchreihenfolge; Teilmengenseed 10 wählt 20 Bilder/Rasse aus
**dem Trainingssplit**. Testdaten beeinflussen weder Training noch Modellwahl.

`probly.transformation.subensemble` baut die tatsächlichen Modelle mit gemeinsamem VGG auf.
Für die Auswertung bereits vorhandener Merkmale wird ein äquivalentes öffentliches probly-
Kopfensemble mit denselben Gewichten genutzt: Der normale Subensemble-Aufruf würde den
Backbone für jedes Mitglied erneut auswerten. MC Dropout nutzt den öffentlichen `Sampler` nur für Kopf 0 und aktiviert
ausschließlich sein bereits trainiertes Dropout; es wird keine zusätzliche Schicht eingefügt. `torch_cpu_context` isoliert und restauriert den Zufallszustand.

In [ ]:
started = time.perf_counter()
conditions = fitted_conditions(TOPIC, data, features, encoder, config)
timings["head_preparation_seconds"] = time.perf_counter() - started
for condition, fitted in conditions.items():
    print(condition, "selected epochs:", [head["best_epoch"] for head in fitted["record"]["heads"]])
    print("  Composed probly subensemble members:", len(fitted["subensemble"]))
print(f"Head preparation: {timings['head_preparation_seconds']:.1f} s")

## 5. Diagnosebilder auswählen

Das `full`-Ensemble sagt für jedes der 3.669 unmaskierten Testbilder eine Rasse vorher.
Als **Modellkonfidenz** verwenden wir die mittlere Wahrscheinlichkeit der vorhergesagten
Rasse. Unter den richtigen Vorhersagen wählen wir die mit der höchsten und die mit der
niedrigsten Konfidenz. Unter den falschen wählen wir die mit der höchsten Konfidenz.
Bei einem Gleichstand entscheidet die alphabetisch erste Bild-ID.

Die Fallnamen beziehen sich auf dieses Modell und seine unkalibrierten Wahrscheinlichkeiten.
Die drei Bilder zeigen gezielt kontrastierende Vorhersagen. Im Modus `seeded` wählt
`select_cases` die Bilder anhand der Rassenlabels und des Seeds aus. In beiden Modi
wird nur für die ausgewählten Bilder die teure Auswertung der Maskierungskombinationen gestartet.

In [ ]:
started = time.perf_counter()
full_ensemble = conditions["full"]["representers"]["ensemble"]
selected_cases, missing_roles = select_cases(
    data, features, full_ensemble, config, mode=COHORT_MODE
)
timings["cohort_selection_seconds"] = time.perf_counter() - started
cohort_ids = [case["image_id"] for case in selected_cases]
print("Selected test images:", cohort_ids)

if missing_roles:
    show_display(Markdown("**Fehlende Diagnosefälle:** " + ", ".join(missing_roles)))

overview = []
captions: dict[str, str] = {}
for case in selected_cases:
    true_class = data["classes"][case["true_class"]]
    predicted_class = data["classes"][case["target_class"]]
    confidence = f"{case['confidence']:.2%}"
    if confidence == "100.00%" and case["confidence"] < 1.0:
        confidence = "≈100%"
    caption = [
        f"Wahr: {true_class}",
        f"Vorhersage: {predicted_class}",
        f"Modellkonfidenz: {confidence}",
    ]
    if case["role"] is not None:
        caption.insert(0, case["role"])
    captions[case["image_id"]] = "\n".join(caption)
    crop = crop_image(data["datasets"]["test"][case["index"]][0])
    overview.append((crop, true_class, case["image_id"]))

for _ in plot_test_overview(overview, captions=captions):
    plt.show()

## 6. Bildspiel und skalare Ansichten derselben Repräsentation

Die Aufteilung in `REGION_MODE` legt fest, welche Pixel gemeinsam sichtbar werden:

- `"trimap"` (Standard) verwendet die Oxford-Labels für Tier, Hintergrund und **nicht
  klassifiziert** (oft eine Übergangszone). Die Maske erhält dieselbe Geometrie wie der
  RGB-Zuschnitt. Nearest Neighbor erhält dabei die ganzzahligen Maskenlabels.
- `"grid"` teilt das Bild in neun feste Rechtecke.
- `"superpixels"` gruppiert ähnliche benachbarte Pixel mit shapiqs SLIC-Helfer zu bis zu
  neun Regionen. Diese Grenzen können auch über die Kontur des Tiers hinweg verlaufen.

Nur vorhandene Regionen zählen als Spieler. Bei drei Regionen gibt es acht Koalitionen.
Die Trimap für `wheaten_terrier_195` markiert den gesamten Zuschnitt als Hintergrund.
In diesem Fall entsteht ein einzelner Shapley-Wert aus zwei Koalitionen. Ein Spieler hat
keine Paarinteraktion.

`MASK_FILL` legt den Ersatz für fehlende Regionen fest. Bei `"blur"` stammt der Ersatz aus
einer unscharfen Kopie. Die Zielrasse für P ist die Vorhersage des **vollständigen Ensembles auf
dem unmaskierten Bild** und bleibt für alle Aufteilungen, Koalitionen und Datenbedingungen fest.
Die wahre Rasse wird zusätzlich dokumentiert.

Zuerst demonstriert `ImageProblyGame` den direkten Bildpfad. Seine öffentliche `masked_image`-
Methode erzeugt anschließend das maskierte Bild für jede Koalition. VGG kodiert jedes einmal. Für die
umfangreichere Auswertung verbindet `ProblyPredictor(preprocess=lookup_features)` die gespeicherten
Merkmale mit `ProblyGame`. Beide Wege werden an leerer und voller Koalition verglichen.

`game.for_output(...)` teilt Repräsentation und Auswertungscache über P/TU/AU/EU. Die Spiele
liefern Rohwerte (`normalize=False`), einschließlich des tatsächlichen leeren Referenzbilds.
probly berechnet Entropie und Zerlegung in **nats**: TU = H(E[p]), AU = E[H(p)], EU = TU − AU.
Es wird kein Entropie- oder Shapley-Algorithmus im Notebook nachgebaut. Form, Wertebereiche,
Zerlegung und Effizienz von SV und k-SII werden für alle Ergebnisse geprüft.

In [ ]:
OUTPUTS: dict[str, str] = {"P": "probability", "TU": "total", "AU": "aleatoric", "EU": "epistemic"}
results: dict[tuple[str, str, str], ExplanationResult] = {}
image_games: dict[str, ImageProblyGame] = {}
region_names: dict[str, tuple[str, ...]] = {}
started = time.perf_counter()


def direct_inputs(images: np.ndarray) -> torch.Tensor:
    return encode_crops(encoder, images, config.feature_batch_size).double()


for case in selected_cases:
    test_index = case["index"]
    image_id = case["image_id"]
    true_class_index = case["true_class"]
    target = case["target_class"]
    image = data["datasets"]["test"][test_index][0]
    crop = crop_image(image)
    regions, region_names[image_id] = image_regions(
        image,
        crop,
        mode=REGION_MODE,
        data_root=TOPIC / "data" if DATA_ROOT is None else DATA_ROOT,
        image_id=image_id,
    )
    direct_predictor = ProblyPredictor(
        full_ensemble,
        preprocess=direct_inputs,
        class_index=target,
        evaluation_context=torch_cpu_context(conditions["full"]["heads"]),
        random_state=config.seed,
    )
    image_game = ImageProblyGame(
        crop, direct_predictor, regions=regions, fill=MASK_FILL, normalize=False
    )
    image_games[image_id] = image_game
    coalitions, table, lookup_features = cached_coalition_features(
        TOPIC, image_game, image_id, encoder, config, output_dir=output_dir
    )
    n_players = image_game.n_players
    indices = {"SV": 1}
    if n_players >= 2:
        indices["k-SII"] = 2
    np.testing.assert_equal(coalitions.shape, (2**n_players, n_players))
    torch.testing.assert_close(table[-1], features["test"][test_index], rtol=1e-4, atol=1e-4)

    for condition, fitted in conditions.items():
        for method, rep in fitted["representers"].items():
            model = fitted["heads"] if method == "ensemble" else fitted["mc_head"]
            predictor = ProblyPredictor(
                rep,
                preprocess=lookup_features,
                class_index=target,
                base=None,
                evaluation_context=torch_cpu_context(model),
                random_state=config.seed,
            )
            game = ProblyGame(n_players, predictor, normalize=False)
            games = {short: game.for_output(name) for short, name in OUTPUTS.items()}
            values = {short: scalar_game(coalitions) for short, scalar_game in games.items()}
            for short, vector in values.items():
                np.testing.assert_equal(vector.shape, (len(coalitions),))
                np.testing.assert_equal(np.isfinite(vector).all(), desired=True)
                upper = 1.0 if short == "P" else float(np.log(len(data["classes"])))
                np.testing.assert_equal(
                    np.all((vector >= -1e-11) & (vector <= upper + 1e-11)), desired=True
                )
            np.testing.assert_allclose(values["TU"], values["AU"] + values["EU"], atol=1e-11)
            exact: dict[str, dict[str, InteractionValues]] = {}
            for short, scalar_game in games.items():
                computer = ExactComputer(scalar_game)
                exact[short] = {
                    index: computer(index=index, order=order) for index, order in indices.items()
                }
                for explanation in exact[short].values():
                    np.testing.assert_equal(np.isfinite(explanation.values).all(), desired=True)
                    nonempty_sum = sum(
                        value
                        for interaction, value in explanation.dict_values.items()
                        if interaction
                    )
                    np.testing.assert_allclose(
                        explanation.baseline_value + nonempty_sum, values[short][-1], atol=1e-9
                    )
                    np.testing.assert_allclose(
                        explanation.baseline_value, values[short][0], atol=1e-11
                    )
            for index in indices:
                total, aleatoric, epistemic = (exact[short][index] for short in ("TU", "AU", "EU"))
                terms = sorted(
                    total.dict_values.keys()
                    | aleatoric.dict_values.keys()
                    | epistemic.dict_values.keys()
                )
                np.testing.assert_allclose(
                    [total[term] for term in terms],
                    [aleatoric[term] + epistemic[term] for term in terms],
                    atol=1e-9,
                )
            results[(image_id, condition, method)] = {
                "target": target,
                "true_class_index": true_class_index,
                "coalitions": coalitions,
                "values": values,
                "exact": exact,
                "games": games,
            }
    reference_run = results[(image_id, "full", "ensemble")]
    np.testing.assert_allclose(
        reference_run["values"]["P"][-1], case["confidence"], rtol=1e-4, atol=1e-5
    )
    cached_game = reference_run["games"]["TU"]
    np.testing.assert_allclose(
        image_game(coalitions[[0, -1]]), cached_game(coalitions[[0, -1]]), rtol=1e-4, atol=1e-5
    )
    print(
        image_id,
        "| players:",
        ", ".join(region_names[image_id]),
        f"| {len(coalitions)} coalitions | fixed target:",
        data["classes"][target],
        "| direct and cached paths agree",
    )
timings["explanation_seconds"] = time.perf_counter() - started

## 7. Vorhersage, Unsicherheit und regionale Interaktionen

Rot bezeichnet positive, Blau negative Beiträge zum jeweiligen Score. Ein positiver EU-Beitrag
erhöht den EU-Spielwert in dieser Erklärung. P-Beiträge haben die Einheit Δp
(0,1 entspricht zehn Prozentpunkten). TU/AU/EU-Beiträge haben die Einheit **Nats**.
Die Zahlen über jeder Karte geben die Scores des maskierten Referenzbilds
und des vollständig sichtbaren Bilds an.

Die Karten zeigen einen Wert pro Region. Das Referenzbild zeigt die Regionsgrenzen.
TU und AU teilen eine Farbskala. P, EU-Shapley-Beiträge und EU-Paarinteraktionen haben jeweils
eigene Skalen. Jede Skala gilt für alle ausgewählten Bilder **des jeweiligen Modus**.
EU-Shapley-Beiträge sind zusätzlich mit Vorzeichen beschriftet.

Die vier Regionskarten zeigen gewöhnliche Shapley-Werte (SV). Bei mindestens zwei Spielern
zeigt die EU-Paarmatrix Beiträge zweiter Ordnung aus k-SII. Auf der grauen Diagonale
liegen Selbstpaare. Ein Wechsel der Aufteilung ändert die Spieler und damit die
erklärte Fragestellung. Die Rohwerte der leeren und vollen Koalition sind weiterhin vergleichbar.

In [ ]:
plot_limits = explanation_plot_limits(results)
for case in selected_cases:
    image_id = case["image_id"]
    plot_region_explanation(
        image_id,
        image_games[image_id],
        results[(image_id, "full", "ensemble")],
        data["classes"],
        plot_limits,
        case_label=case["role"],
        region_mode=REGION_MODE,
        region_names=region_names[image_id],
    )
    plt.show()

## 8. Weniger Trainingsdaten und MC Dropout vergleichen

Dieselben Merkmale und dieselbe Zielklasse werden unter beiden Trainingsbedingungen ausgewertet.
Die EU-Karten aller Bilder und Varianten nutzen dieselbe Farbskala wie die EU-Shapley-Karte
in Abschnitt 7 innerhalb derselben Regionsaufteilung. Vorzeichen und Zahlen machen auch kleine Beiträge lesbar. Die Werte über den
Karten vergleichen das maskierte Referenzbild mit dem vollständig sichtbaren Bild.

Der Backbone bleibt gleich. Die Unterschiede stammen aus den trainierten Köpfen und der
Samplingmethode. Für MC Dropout sind Seeds und 64 Samples fest. Wiederholte Koalitionsaufrufe
definieren damit dasselbe Monte-Carlo-Spiel. Die Karten sind lokale Erklärungen für diese
Modellläufe und die gewählte Maskierung. Ein EU-Anstieg bei weniger Trainingsdaten ist
keine garantierte Eigenschaft. Die Tabelle nennt wahre Rasse und feste P-Zielklasse ausdrücklich.

In [ ]:
summary: list[SummaryRow] = []
for case in selected_cases:
    image_id = case["image_id"]
    runs: dict[tuple[str, str], ExplanationResult] = {
        (condition, method): results[(image_id, condition, method)]
        for condition in conditions
        for method in ("ensemble", "mc_dropout")
    }
    plot_epistemic_comparison(
        image_id,
        image_games[image_id],
        runs,
        plot_limits,
        case_label=case["role"],
        region_mode=REGION_MODE,
        region_names=region_names[image_id],
    )
    plt.show()
    for (condition, method), result in runs.items():
        summary.append(
            {
                "image_id": image_id,
                "condition": condition,
                "method": method,
                "true_class": data["classes"][result["true_class_index"]],
                "target": data["classes"][result["target"]],
                "target_index": result["target"],
                "P": float(result["values"]["P"][-1]),
                "TU": float(result["values"]["TU"][-1]),
                "AU": float(result["values"]["AU"][-1]),
                "EU": float(result["values"]["EU"][-1]),
            }
        )

rows = [
    "| Image | True class | Fixed P target | Condition | Method | P [p] | TU [nats] | AU [nats] | EU [nats] |",
    "| :--- | :--- | :--- | :--- | :--- | ---: | ---: | ---: | ---: |",
]
rows.extend(
    f"| {row['image_id']} | {row['true_class']} | {row['target']} | {row['condition']} | {row['method']} | "
    + " | ".join(f"{row[key]:.5g}" for key in ("P", "TU", "AU", "EU"))
    + " |"
    for row in summary
)
show_display(Markdown("\n".join(rows)))

## 9. Ergebnisse und Reproduzierbarkeit

TU = AU + EU sowie die entsprechende Linearität und Effizienz von SV/k-SII sind technische
Konsistenzprüfungen, keine empirischen Entdeckungen. AU/EU bezeichnen eine modellbasierte
Zerlegung. Gemeinsame Merkmale begrenzen das Ensemble auf Unsicherheit seiner Köpfe;
Maskierungen mit Blur können selbst Verteilungsverschiebungen erzeugen. Diese erste Fallstudie
ist keine neue unabhängige Validierung der Erklärungsgüte.

`summary.json` enthält Konfiguration, tatsächliche Hardware-/Laufzeitdaten, offizielle probly-
Quelle, Merkmalsherkunft, Auswahlregel, Diagnosefälle sowie die skalaren Ergebnisse.
Die Zielklasse und die Konfidenz jedes Diagnosefalls beziehen sich auf das `full`-Ensemble.
Die Konfidenz wird mit dem P-Spielwert der vollen Koalition verglichen.
Jedes NumPy-Archiv enthält zusätzlich die explizite Koalitionsreihenfolge, Regionskarte,
Klassenindizes, Einheiten, Baselines, den Regionsmodus und die geordneten Spielernamen. SV und k-SII werden außerdem mit shapiqs vorhandenem
`InteractionValues.to_json_file` gespeichert; damit bleiben Index, Ordnung und Interaktionszuordnung
erhalten. Raster-Ergebnisse bleiben in `outputs/`. Trimap und Superpixel schreiben ihre
Koalitionscaches und Exporte nach `outputs/trimap/` beziehungsweise `outputs/superpixels/`.
Alle Exporte werden nach dem Schreiben wieder geladen und geprüft.

In [ ]:
exported_explanations = export_explanations(
    output_dir,
    results,
    image_games,
    data["classes"],
    mask_fill=MASK_FILL,
    region_mode=REGION_MODE,
    region_names=region_names,
)
timings["total_seconds"] = time.perf_counter() - run_started
report = {
    "config": asdict(config),
    "versions": versions,
    "runtime": asdict(runtime),
    "timings_seconds": timings,
    "feature_provenance": features["provenance"],
    "cohort_ids": cohort_ids,
    "mask_fill": MASK_FILL,
    "region_mode": REGION_MODE,
    "region_names": region_names,
    "superpixel_target": SUPERPIXEL_TARGET if REGION_MODE == "superpixels" else None,
    "cohort_selection": {
        "mode": COHORT_MODE,
        "reference": "full/ensemble",
        "candidate_count": len(features["test"]) if COHORT_MODE == "demo" else len(selected_cases),
        "rule": (
            "Highest and lowest top-1 probability among correct predictions, highest among errors; "
            "ties by image ID."
            if COHORT_MODE == "demo"
            else "Seeded label-only selection with at most one image per breed."
        ),
        "missing_roles": missing_roles,
        "cases": [
            {
                **case,
                "true_class_name": data["classes"][case["true_class"]],
                "target_class_name": data["classes"][case["target_class"]],
            }
            for case in selected_cases
        ],
    },
    "classes": data["classes"],
    "summary": summary,
    "explanation_files": exported_explanations,
    "scope": "descriptive image-region games; fixed VGG features; raw nats",
}
export_summary(output_dir, report, expected_entries=len(results))
print(
    f"Checked {len(results)} NumPy archives and {len(exported_explanations)} explanation JSON files."
)
print("Local results:", output_dir)
print(f"Total notebook runtime: {timings['total_seconds'] / 60:.1f} min")

### Quellen

- [probly: gemeinsame Pipeline](https://pwhofman.github.io/probly/latest/guide/core_pillars.html)
- [probly: Second-Order-Methoden](https://pwhofman.github.io/probly/latest/guide/methods/second_order.html)
- [Oxford-IIIT Pet: Daten und Originalarbeit](https://www.robots.ox.ac.uk/~vgg/data/pets/)
- [torchvision VGG16 und Gewichte](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.vgg16.html)

Der Vergleich baut auf dem ursprünglichen `dan_02-3_oxford-pets-probly-shapiq.ipynb` auf.
Er verwendet die integrierten Klassen statt eigener Score-/Game-Implementierungen, ergänzt
MC Dropout und ist ohne dessen Geschwister-Notebooks lauffähig. Die P-Zielklasse wird hier
explizit aus dem vollständigen Ensemble gewählt; das Original verwendete den einzelnen Baseline-Kopf.